# BIOMED UMSS — Clasificador de cromosomas v3 (EfficientNet-B3)

Fase C6 (ADR-0007 / DD-ML-001). Parte del **v2 ya validado** (val_acc 0.6771 /
macro-F1 0.6517, split honesto por cariograma) y ataca los tres problemas que
revela su reporte por clase.

## Lo que el v2 hizo bien — y no se toca
El F1 por **grupo de Denver** (la clasificación citogenética estándar por tamaño)
muestra un gradiente limpio:

| grupo | F1 |
|---|---|
| A (1-3), los más grandes | **0.85** |
| D (13-15) | 0.69 |
| E (16-18) | 0.68 |
| C (6-12) | 0.67 |
| F (19-20) | 0.66 |
| B (4-5) | 0.65 |
| G (21-22), los más chicos | **0.54** |

Cuanto más grande el cromosoma, mejor lo clasifica: **el letterbox funcionó**, la
señal de tamaño relativo llega a la red. El split por cariograma y el letterbox se
mantienen exactamente igual que en el v2.

## Los tres cambios del v3

### 1. Sampler suavizado — el problema más grande
El v2 sobre-predice X e Y de forma severa. Con precisión 0.14 y recall 0.43 sobre
58 casos reales, el Y da **25 aciertos y ~153 falsos positivos**: por cada Y
correcto, el modelo grita «Y» seis veces de más. El X: 59 aciertos, ~139 falsos
positivos.

No es falta de datos, es **calibración**. El `WeightedRandomSampler` con pesos
`1/frecuencia` sobre-muestrea tanto las clases raras que la red aprende «ante la
duda, decí X o Y». v3 usa **`1/sqrt(frecuencia)`**: sigue compensando el
desbalance, sin exagerarlo.

*Impacto medido:* sin X ni Y el macro-F1 del v2 sería 0.685 en vez de 0.652 — esas
dos clases solas cuestan 3.4 puntos.

### 2. Resolución nativa (224 → 300 px)
EfficientNet-B3 fue diseñada para **300 px**; el v2 entrenó a 224. Los cromosomas
21 y 22 miden ~29 px de alto: dentro de un lienzo de 224 con letterbox ocupan
poquísimo. Más resolución da más píxeles justo donde el modelo falla (grupos G y C).

### 3. Más fine-tune + augmentación menos agresiva
La mejor época del v2 fue la **7 de 8** del fine-tune: seguía mejorando al final,
o sea que **no convergió**. v3 sube a 12 épocas.

Además baja la rotación de 25° a 15°. Cuando el tamaño ya no alcanza para
distinguir (grupos C y G), lo que usa un citogenetista es el **patrón de bandas**;
una rotación agresiva lo difumina por interpolación.

> ⚠️ **300 px cuesta ~1.8× más cómputo por imagen que 224.** Con 48.467 crops y 18
> épocas, calcular **3-4 h en una T4**. Si la cuota de GPU es un problema, bajá
> `EPOCHS_HEAD` a 4 o volvé a `IMG_SIZE = 224` — los otros dos cambios ya aportan
> por sí solos.


In [ ]:
# 1) Setup
import os, re, json, time, zipfile, random, copy
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| device:', device)
if device == 'cpu':
    print('AVISO: sin GPU -> lento. Activa GPU en el runtime.')

In [ ]:
# 2) Config
IMG_SIZE     = 300    # v3: resolucion nativa de EfficientNet-B3 (v2 usaba 224)
BATCH        = 32     # v3: mitad, porque 300px ocupa ~1.8x mas memoria
EPOCHS_HEAD  = 6      # fase 1: cabeza (backbone congelado)
EPOCHS_FT    = 12     # v3: el v2 mejoraba aun en su ultima epoca (no convergio)
VAL_FRAC     = 0.15   # fraccion de CARIOGRAMAS (no de crops)
TARGET_FRAC  = 0.45   # un cromosoma de altura mediana ocupa 45% del lienzo
FILL         = 255    # fondo blanco (medido: 79% de las esquinas de los crops >= 240)
DATA_DIR     = 'crops'
OUT_DIR      = 'model_out'
os.makedirs(OUT_DIR, exist_ok=True)

# Copia de seguridad del mejor checkpoint en Drive (sobrevive a desconexiones).
# Vacío ('') lo desactiva.
CKPT_DIR = '/content/drive/MyDrive/biomed_ckpt_v3'

# v3: exponente del sampler. 1.0 = 1/frecuencia (v2, sobre-corrige);
# 0.5 = 1/sqrt(frecuencia) (compensa sin exagerar); 0 = sin balanceo.
SAMPLER_POWER = 0.5
ROTACION_GRADOS = 15  # v3: 25 difuminaba el bandeo por interpolacion

## Cargar el dataset desde Google Drive

`crops.zip` son ~84 MB y 48.467 archivos. **No uses `files.upload()`**: en Colab
ese diálogo es lento y se corta seguido. Súbelo a tu Drive una vez y móntalo — si
el entorno se desconecta, el archivo sigue ahí y no hay que resubir nada.

Sube `datasets/metaclass/crops.zip` a tu Drive (cualquier carpeta sirve) y corre
la celda siguiente: **busca el archivo sola**. Si no lo encuentra, te lista los
`.zip` que sí hay para que copies la ruta correcta en `ZIP_EN_DRIVE`.

> ⚠️ `drive.mount()` recibe el **punto de montaje del runtime**, no una carpeta de
> tu Drive: siempre `'/content/drive'`. Tu Drive queda bajo
> `/content/drive/MyDrive/`, así que un archivo en *Colab Notebooks* está en
> `/content/drive/MyDrive/Colab Notebooks/crops.zip`. Cambiar el punto de montaje
> es la causa más común de que "no reconozca el Drive".


In [ ]:
# 3) Cargar crops.zip desde Google Drive
# OJO: drive.mount() recibe el PUNTO DE MONTAJE del runtime, no una carpeta de
# tu Drive. Siempre '/content/drive'. Tu Drive queda bajo /content/drive/MyDrive/.
ZIP_EN_DRIVE = ''   # vacío = buscarlo solo en el Drive

if not os.path.isdir(DATA_DIR):
    from google.colab import drive
    drive.mount('/content/drive')

    MI_UNIDAD = '/content/drive/MyDrive'
    if not ZIP_EN_DRIVE:
        # Busca crops.zip en cualquier carpeta del Drive (raíz, 'Colab
        # Notebooks', etc.) para no depender de acertar la ruta a mano.
        print('Buscando crops.zip en tu Drive...')
        for base, _, archivos in os.walk(MI_UNIDAD):
            if 'crops.zip' in archivos:
                ZIP_EN_DRIVE = os.path.join(base, 'crops.zip')
                break

    if not ZIP_EN_DRIVE or not os.path.exists(ZIP_EN_DRIVE):
        print('No se encontro crops.zip. Archivos .zip en tu Drive:')
        for base, _, archivos in os.walk(MI_UNIDAD):
            for a in archivos:
                if a.endswith('.zip'):
                    print('  ', os.path.join(base, a))
        raise FileNotFoundError(
            'Sube crops.zip a tu Drive, o pon su ruta completa en ZIP_EN_DRIVE '
            '(empieza con /content/drive/MyDrive/).')

    print('Usando:', ZIP_EN_DRIVE)
    # Descomprimir en el disco local del runtime (NO en Drive): leer 48k archivos
    # por FUSE en cada época es mucho más lento que desde el disco local.
    print('Descomprimiendo...')
    with zipfile.ZipFile(ZIP_EN_DRIVE) as z:
        z.extractall(DATA_DIR)
    print('Listo')

# el zip trae un nivel extra (crops/crops/1/...)
_inner = os.path.join(DATA_DIR, 'crops')
if os.path.isdir(_inner) and not os.path.isdir(os.path.join(DATA_DIR, '1')):
    DATA_DIR = _inner

if CKPT_DIR:
    os.makedirs(CKPT_DIR, exist_ok=True)
    print('checkpoints ->', CKPT_DIR)

assert os.path.isdir(DATA_DIR), f'No existe {DATA_DIR}'
print('DATA_DIR =', DATA_DIR)
print('clases:', sorted(os.listdir(DATA_DIR)))
print('crops:', sum(len(f) for _, _, f in os.walk(DATA_DIR)))


In [ ]:
# 4) Indexar crops + agrupar por CARIOGRAMA (la unidad de paciente)
#    El nombre del crop codifica su origen: '<clase>/cario<ID>_<idx>.png'.
#    Verificado sobre los 19.845 crops del manifiesto: 100% derivable.
CLASSES = [str(n) for n in range(1, 23)] + ['X', 'Y']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
PAT = re.compile(r'^cario(\d+)_\d+\.png$')

samples = []          # (path, label_idx, cariograma_id)
skipped = 0
for cls in sorted(os.listdir(DATA_DIR)):
    cdir = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cdir) or cls not in CLASS_TO_IDX:
        continue
    for fn in sorted(os.listdir(cdir)):
        m = PAT.match(fn)
        if not m:
            skipped += 1
            continue
        samples.append((os.path.join(cdir, fn), CLASS_TO_IDX[cls], m.group(1)))

groups = sorted({g for _, _, g in samples})
print(f'crops: {len(samples)} | cariogramas: {len(groups)} | sin parsear: {skipped}')
assert len(samples) > 0, 'no se indexo ningun crop'
assert len(groups) > 10, 'muy pocos cariogramas: revisa el naming de los archivos'

In [ ]:
# 5) Escala de referencia POR CARIOGRAMA (altura mediana de sus cromosomas).
#    Es lo que hace la escala invariante al zoom del microscopio conservando
#    que el 1 es ~2x la mediana y el 21 ~0.4x. Inferencia calcula lo mismo
#    sobre las detecciones de la metafase (app/preprocess.reference_height).
heights = defaultdict(list)
t0 = time.time()
for path, _, g in samples:
    with Image.open(path) as im:
        heights[g].append(im.size[1])   # (w, h)
REF_H = {g: float(np.median(v)) for g, v in heights.items()}
print(f'ref_h calculado para {len(REF_H)} cariogramas ({time.time()-t0:.0f}s)')
print('ejemplo ref_h:', dict(list(REF_H.items())[:5]))

In [ ]:
# 6) SPLIT POR CARIOGRAMA  <-- la correccion del defecto 1
#    Ningun paciente puede estar en train y val a la vez.
rng = random.Random(SEED)
shuffled = groups[:]
rng.shuffle(shuffled)
n_val_groups = max(1, int(len(shuffled) * VAL_FRAC))
val_groups = set(shuffled[:n_val_groups])
train_groups = set(shuffled[n_val_groups:])

train_samples = [s for s in samples if s[2] in train_groups]
val_samples   = [s for s in samples if s[2] in val_groups]

# invariante critico: cero solapamiento de pacientes
assert not (train_groups & val_groups), 'FUGA: un cariograma esta en ambos splits'
assert {s[2] for s in train_samples}.isdisjoint({s[2] for s in val_samples})

print(f'train: {len(train_samples)} crops / {len(train_groups)} cariogramas')
print(f'val:   {len(val_samples)} crops / {len(val_groups)} cariogramas')
vc = np.bincount([s[1] for s in val_samples], minlength=len(CLASSES))
print('soporte val por clase:', {CLASSES[i]: int(v) for i, v in enumerate(vc)})

In [ ]:
# 7) Letterbox  <-- la correccion del defecto 2
#    ESPEJO EXACTO de backend-ml/app/preprocess.py. Si tocas uno, toca el otro:
#    divergir aqui degrada la precision en produccion sin ningun sintoma visible.
def letterbox(img: Image.Image, ref_h: float, canvas: int = IMG_SIZE,
              target_frac: float = TARGET_FRAC, fill: int = FILL) -> Image.Image:
    w, h = img.size
    if h < 1 or w < 1:
        return Image.new('L', (canvas, canvas), fill)
    scale = (target_frac * canvas) / ref_h if ref_h and ref_h > 0 else canvas / max(h, w)
    scale = min(scale, canvas / float(max(h, w)))      # nunca desbordar el lienzo
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    out = Image.new('L', (canvas, canvas), fill)
    out.paste(img.resize((new_w, new_h), Image.BILINEAR),
              ((canvas - new_w) // 2, (canvas - new_h) // 2))
    return out

# Chequeo visual: el 1 debe verse claramente mas grande y alargado que el 21.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, cls in zip(axes, ['1', '9', '21', 'X']):
    s = next(x for x in samples if CLASSES[x[1]] == cls)
    with Image.open(s[0]) as im:
        ax.imshow(letterbox(im.convert('L'), REF_H[s[2]]), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'clase {cls}'); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 8) Dataset + augmentacion
#    La augmentacion va DESPUES del letterbox y NO altera la escala:
#    el tamano relativo ahora es una feature, escalarlo al azar la destruiria.
#    Rotacion/flip si: la orientacion de un cromosoma en la lamina es arbitraria.
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]

train_aug = transforms.Compose([
    transforms.RandomRotation(ROTACION_GRADOS, fill=FILL),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAffine(0, translate=(0.04, 0.04), fill=FILL),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_aug = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class ChromosomeDS(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, y, g = self.items[i]
        with Image.open(path) as im:
            canvas = letterbox(im.convert('L'), REF_H.get(g, 0.0))
        return self.tf(canvas), y

train_data = ChromosomeDS(train_samples, train_aug)
val_data   = ChromosomeDS(val_samples, eval_aug)

# v3: balanceo SUAVIZADO  <-- la correccion del problema de X e Y
#
# El v2 usaba 1/frecuencia. Con Y en ~359 crops contra ~2200 de las clases 1-18,
# eso lo sobre-muestreaba ~6x: la red aprendio "ante la duda, deci X o Y" y
# termino con Y en precision 0.14 / recall 0.43 (25 aciertos, ~153 falsos
# positivos). Es sobre-correccion del sampler, no falta de datos.
#
# 1/frecuencia^0.5 sigue compensando el desbalance pero mucho mas suave.
targets = [s[1] for s in train_samples]
counts = np.bincount(targets, minlength=len(CLASSES))
class_w = 1.0 / np.power(np.clip(counts, 1, None), SAMPLER_POWER)
sampler = WeightedRandomSampler([class_w[t] for t in targets], len(targets), replacement=True)

# Cuantas veces se ve la clase mas rara vs la mas comun, por epoca:
_pmin, _pmax = class_w[counts > 0].min(), class_w.max()
print(f'sampler: power={SAMPLER_POWER} -> sobre-muestreo maximo {_pmax/_pmin:.1f}x '
      f'(con power=1.0 seria {counts[counts>0].max()/counts[counts>0].min():.1f}x)')

train_loader = DataLoader(train_data, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_data, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('conteo train por clase:', {CLASSES[i]: int(c) for i, c in enumerate(counts)})

In [ ]:
# 9) Modelo: EfficientNet-B3 pre-entrenado + cabeza de 24 clases
weights = models.EfficientNet_B3_Weights.IMAGENET1K_V1
model = models.efficientnet_b3(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASSES))
model = model.to(device)

def set_backbone_trainable(flag):
    for p in model.features.parameters():
        p.requires_grad = flag

# label smoothing: las etiquetas de la fila 4 (19-22/X/Y) son best-effort (C1),
# asi que conviene no penalizar al 100% la confianza en ellas.
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

In [ ]:
# 10) Entrenamiento + evaluacion (guarda el MEJOR checkpoint, no el ultimo)
def evaluate():
    model.eval(); correct = total = 0
    all_y, all_p = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item(); total += y.size(0)
            all_y += y.cpu().tolist(); all_p += pred.cpu().tolist()
    f1s = []
    for c in range(len(CLASSES)):
        tp = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp == c)
        fp = sum(1 for yy, pp in zip(all_y, all_p) if yy != c and pp == c)
        fn = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp != c)
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec  = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return correct / total, float(np.mean(f1s)), all_y, all_p

BEST = {'f1': -1.0, 'acc': 0.0, 'state': None, 'phase': None, 'epoch': None}

def train_phase(name, epochs, lr, train_backbone):
    set_backbone_trainable(train_backbone)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model.train(); t0 = time.time(); running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step(); running += loss.item()
        sched.step()
        acc, f1, *_ = evaluate()
        flag = ''
        if f1 > BEST['f1']:
            BEST.update(f1=f1, acc=acc, phase=name, epoch=ep + 1,
                        state=copy.deepcopy(model.state_dict()))
            flag = '  <-- mejor'
            # Persistir el mejor checkpoint FUERA del runtime. Colab se
            # desconecta (inactividad, cuota de GPU) y se lleva el disco local
            # con él; sin esto, una caída a mitad del fine-tune cuesta el
            # entrenamiento entero. Se escribe solo al mejorar, no cada época.
            if CKPT_DIR:
                torch.save(model.state_dict(), f'{CKPT_DIR}/classifier_best.pth')
                json.dump({'phase': name, 'epoch': ep + 1,
                           'val_macro_f1': round(f1, 4), 'val_accuracy': round(acc, 4)},
                          open(f'{CKPT_DIR}/checkpoint_meta.json', 'w'), indent=2)
        print(f'  [{name}] ep {ep+1}/{epochs}  loss={running/len(train_loader):.3f}  '
              f'val_acc={acc:.3f}  val_macroF1={f1:.3f}  ({time.time()-t0:.0f}s){flag}')

In [ ]:
# 11) Fase 1 — cabeza (backbone congelado)
print('== Fase 1: entrenar la cabeza ==')
train_phase('head', EPOCHS_HEAD, lr=1e-3, train_backbone=False)

In [ ]:
# 12) Fase 2 — fine-tune del backbone (LR bajo)
print('== Fase 2: fine-tune ==')
train_phase('ft', EPOCHS_FT, lr=1e-4, train_backbone=True)

In [ ]:
# 13) Restaurar el MEJOR checkpoint y reportar por clase
assert BEST['state'] is not None, 'no se entreno ninguna epoca'
model.load_state_dict(BEST['state'])
acc, f1, all_y, all_p = evaluate()
print(f"MEJOR: fase={BEST['phase']} epoca={BEST['epoch']}")
print(f'ACCURACY={acc:.4f}  MACRO-F1={f1:.4f}')
print(f'  v2 (mismo split honesto): 0.6771 / 0.6517  <-- ESTA es la comparacion valida')
print(f'  v1 (split CON FUGA):      0.6334 / 0.6008  <-- inflado, no comparable')
print(f'  delta macro-F1 vs v2: {f1 - 0.6517:+.4f}')
print('\nPor clase (precision/recall/soporte):')
for c in range(len(CLASSES)):
    tp = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp == c)
    fp = sum(1 for yy, pp in zip(all_y, all_p) if yy != c and pp == c)
    fn = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp != c)
    sup = sum(1 for yy in all_y if yy == c)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    print(f'  {CLASSES[c]:>2}: P={prec:.2f} R={rec:.2f} n={sup}')

In [ ]:
# 14-bis) Diagnostico v3: se corrigio la sobre-prediccion de X e Y?
#
# El sintoma en el v2 era recall >> precision en X e Y: el modelo los gritaba de
# mas. Si el sampler suavizado funciono, la precision sube y los falsos positivos
# bajan (el recall puede ceder algo — es el intercambio buscado).
V2 = {'X': (0.30, 0.41), 'Y': (0.14, 0.43)}   # (precision, recall) del v2

print(f'{"cls":>3} {"P":>6} {"R":>6} {"F1":>6} {"FP":>6}   vs v2')
for cls in ['X', 'Y']:
    c = CLASSES.index(cls)
    tp = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp == c)
    fp = sum(1 for yy, pp in zip(all_y, all_p) if yy != c and pp == c)
    fn = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp != c)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1c = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    p2, r2 = V2[cls]
    print(f'{cls:>3} {prec:>6.2f} {rec:>6.2f} {f1c:>6.2f} {fp:>6}   '
          f'v2: P={p2:.2f} R={r2:.2f} -> P {prec-p2:+.2f}')

# Macro-F1 con y sin X/Y: cuanto pesan todavia
f1s = []
for c in range(len(CLASSES)):
    tp = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp == c)
    fp = sum(1 for yy, pp in zip(all_y, all_p) if yy != c and pp == c)
    fn = sum(1 for yy, pp in zip(all_y, all_p) if yy == c and pp != c)
    pr = tp / (tp + fp) if tp + fp else 0.0
    rc = tp / (tp + fn) if tp + fn else 0.0
    f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
sin_xy = [v for i, v in enumerate(f1s) if CLASSES[i] not in ('X', 'Y')]
print()
print(f'macro-F1 completo: {np.mean(f1s):.4f} | sin X/Y: {np.mean(sin_xy):.4f} '
      f'| X e Y cuestan {np.mean(sin_xy) - np.mean(f1s):.3f} (v2: 0.034)')

# F1 por grupo de Denver: el gradiente por tamano debe seguir limpio
GRUPOS = {'A (1-3)': ['1','2','3'], 'B (4-5)': ['4','5'],
          'C (6-12)': ['6','7','8','9','10','11','12'], 'D (13-15)': ['13','14','15'],
          'E (16-18)': ['16','17','18'], 'F (19-20)': ['19','20'], 'G (21-22)': ['21','22']}
print()
print('F1 por grupo de Denver (v2 entre parentesis):')
V2_G = {'A (1-3)': 0.85, 'B (4-5)': 0.65, 'C (6-12)': 0.67, 'D (13-15)': 0.69,
        'E (16-18)': 0.68, 'F (19-20)': 0.66, 'G (21-22)': 0.54}
for g, cs in GRUPOS.items():
    m = np.mean([f1s[CLASSES.index(c)] for c in cs])
    print(f'  {g:<10} {m:.2f}   (v2 {V2_G[g]:.2f})')


In [ ]:
# 14) Matriz de confusion — que clases se confunden (grupos de tamano similar)
cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for yy, pp in zip(all_y, all_p):
    cm[yy, pp] += 1
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap='viridis')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=90, fontsize=8)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
ax.set_xlabel('predicho'); ax.set_ylabel('real'); ax.set_title('Matriz de confusion (val)')
fig.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
# 15) Guardar (para Fase C3 en backend-ml)
#     'preprocess': 'letterbox' es LOAD-BEARING: app/efficientnet.py lo lee para
#     decidir el preprocesamiento. Sin ese campo asume el 'resize' del v1.
torch.save(model.state_dict(), f'{OUT_DIR}/classifier.pth')
json.dump(CLASSES, open(f'{OUT_DIR}/classes.json', 'w'))
json.dump({
    'arch': 'efficientnet_b3',
    'img_size': IMG_SIZE,
    'num_classes': len(CLASSES),
    'normalization': {'mean': IMAGENET_MEAN, 'std': IMAGENET_STD},
    'preprocess': 'letterbox',
    'letterbox': {'target_frac': TARGET_FRAC, 'fill': FILL},
    'split': 'by_karyogram',
    'val_accuracy': round(acc, 4),
    'val_macro_f1': round(f1, 4),
    'val_karyograms': len(val_groups),
    'train_karyograms': len(train_groups),
    'sampler_power': SAMPLER_POWER,
    'rotation_deg': ROTACION_GRADOS,
    'version': 'v3',
}, open(f'{OUT_DIR}/model_meta.json', 'w'), indent=2)
print('guardado en', OUT_DIR, os.listdir(OUT_DIR))

In [ ]:
# 16) Guardar en Drive + descargar
# Drive primero: la descarga del navegador puede fallar o cancelarse, y si el
# runtime muere después, el modelo entrenado se pierde. En Drive queda a salvo.
ARCHIVOS = ['classifier.pth', 'classes.json', 'model_meta.json']

if CKPT_DIR:
    import shutil
    for f in ARCHIVOS:
        shutil.copy(f'{OUT_DIR}/{f}', f'{CKPT_DIR}/{f}')
    print('guardado en Drive:', CKPT_DIR)

try:
    from google.colab import files
    for f in ARCHIVOS:
        files.download(f'{OUT_DIR}/{f}')
except Exception as e:
    print('Descarga directa no disponible; toma los archivos de Drive.', e)


## Siguiente paso
Poné `classifier.pth`, `classes.json` y `model_meta.json` en `backend-ml/models/`.
`app/efficientnet.py` lee `preprocess: 'letterbox'` del meta y aplica el mismo
letterbox de `app/preprocess.py`, calculando `ref_h` sobre las detecciones de la
metafase — sin tocar la API ni el pipeline (hexagonal, ADR-0007).

**Cómo leer el resultado:** compará el macro-F1 v2 contra el v1 *sabiendo que no
son comparables* — el 0.6008 del v1 se midió con fuga de datos. La cifra v2 es la
primera medida honesta sobre pacientes no vistos. Si querés el número comparable,
reentrená el v1 con el split por cariograma: esa es la línea base real.

**Si el macro-F1 sigue bajo en clases concretas**, mirá la matriz de confusión
(celda 14). Confusión dentro de grupos de tamaño similar (19/20/21/22, o 4/5) es
esperable y es exactamente lo que el HITL de RN-01 cubre: esos cromosomas caen en
naranja y el analista los valida. Confusión entre grupos lejanos (1 ↔ 21) indicaría
que la señal de escala no está llegando — revisá la celda 7.